# BaseLineGAT 训练（基因顺序严格对齐）

- 先加载表达数据拿到 target_genes 顺序
- 再用同一份 target_genes 构建图的 node_list（缺边基因自动变为孤立点+自环）
- 从源头避免 graph/expression 错位


In [1]:
import os
import sys
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

np.random.seed(42)
tf.random.set_seed(42)

ROOT = "/Users/liuxi/Desktop/RFA_GNN"
if not os.path.exists(ROOT):
    ROOT = os.getcwd()
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from data_loader import load_rfa_data, build_combined_gnn
from base_gnn import BaseLineGAT

print("ROOT=", ROOT)


ROOT= /Users/liuxi/Desktop/RFA_GNN


In [2]:
use_landmark_genes = True
cell_line = None
max_samples = 2000
epochs = 5
batch_size = 32

tf_path = os.path.join(ROOT, "data/omnipath/omnipath_tf_regulons.csv")
ppi_path = os.path.join(ROOT, "data/omnipath/omnipath_interactions.csv")
string_path = os.path.join(ROOT, "data/string_interactions_mapped.csv")
full_gene_path = os.path.join(ROOT, "data/GSE92742_Broad_LINCS_gene_info.txt")
siginfo_path = os.path.join(ROOT, "data/siginfo_beta.txt")
landmark_path = os.path.join(ROOT, "data/landmark_genes.json")
target_path = os.path.join(ROOT, "data/compound_targets.txt")
ctl_path = os.path.join(ROOT, "data/cmap/level3_beta_ctl_n188708x12328.h5")
trt_path = os.path.join(ROOT, "data/cmap/level3_beta_trt_cp_n1805898x12328.h5")


In [ ]:
data = load_rfa_data(
    ctl_path,
    trt_path,
    landmark_path=landmark_path,
    drug_target_path=target_path,
    siginfo_path=siginfo_path,
    use_landmark_genes=use_landmark_genes,
    full_gene_path=full_gene_path,
    max_samples=max_samples,
    cell_lines=cell_line,
)
if data is None:
    raise RuntimeError("load_rfa_data returned None")

print("X_ctl:", np.asarray(data["X_ctl"]).shape)
print("y_delta:", np.asarray(data["y_delta"]).shape)
print("X_drug:", np.asarray(data["X_drug"]).shape)
print("genes:", len(data["target_genes"]))


正在加载 RFA 数据 (Landmark Mode: True)...
CSV路径: CTL=/Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_ctl_n188708x12328.h5, TRT=/Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_trt_cp_n1805898x12328.h5
正在加载全基因映射: /Users/liuxi/Desktop/RFA_GNN/data/GSE92742_Broad_LINCS_gene_info.txt
目标基因数: 978
正在读取元数据: /Users/liuxi/Desktop/RFA_GNN/data/siginfo_beta.txt
过滤后元数据记录数: 117284
已构建 Control 查找表，覆盖 401 个 (Cell, Time, Batch) 组合。
>>> 进入 DeepCOP 对齐模式: 计算 Cell-Specific Mean Control
发现 57929 个 Control 样本，涉及 162 个细胞系。
正在加载所有 Control 数据以计算均值...
正在加载数据文件: /Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_ctl_n188708x12328.h5 ...
  H5 (h5py): 匹配到 57929 个样本 (axis0/Index). 读取中...
  读取耗时: 19.32s
  Gene ID 匹配: Index=0, Columns=978
  形状检查: Rows=57929, Target=978
  已筛选并对齐到 978 个 Genes。
已计算 162 个细胞系的基准表达谱。
发现 67663 个高质量 Treatment 样本。
正在加载 Treatment 数据...
正在加载数据文件: /Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_trt_cp_n1805898x12328.h5 ...
  H5 (h5py): 匹配到 67663 个样本 (axis0/Index). 读取中...
  读取耗时: 28.38s
  Gene 

In [4]:
symbol_to_entrez = data["symbol_to_entrez"]
adj_matrix, node_list, gene2idx, edge_index = build_combined_gnn(
    tf_path=tf_path,
    ppi_path=ppi_path,
    string_path=string_path,
    target_genes=data["target_genes"],
    confid_threshold=0.9,
    directed=False,
    symbol_to_entrez=symbol_to_entrez,
)
if len(node_list) != len(data["target_genes"]) or node_list[:50] != data["target_genes"][:50]:
    raise ValueError("Graph node_list 与表达 target_genes 顺序/长度不一致")
print("graph nodes:", len(node_list), "edges:", int(edge_index.shape[1]))


>>> 正在构建 Combined GNN (TF + PPI) ...
正在加载全基因映射: /Users/liuxi/Desktop/RFA_GNN/data/GSE92742_Broad_LINCS_gene_info.txt
加载 TF 调控网络: /Users/liuxi/Desktop/RFA_GNN/data/omnipath/omnipath_tf_regulons.csv
TF 边数: 1276
加载 PPI 网络: /Users/liuxi/Desktop/RFA_GNN/data/omnipath/omnipath_interactions.csv
PPI 边数: 1605
加载 STRING 网络: /Users/liuxi/Desktop/RFA_GNN/data/string_interactions_mapped.csv
STRING 添加的边数: 1930
  STRING 边数: 1930
Combined Graph 构建完成: 978 节点, 3270 边 (含权重)
graph nodes: 978 edges: 5285


In [5]:
le = LabelEncoder()
cell_idx = le.fit_transform(data["cell_names"])
num_cells = int(len(le.classes_))

drug_ids = np.asarray(data["drug_ids"], dtype=str)
unique_drugs = np.unique(drug_ids)
np.random.seed(42)
test_drugs = np.random.choice(unique_drugs, int(len(unique_drugs) * 0.2), replace=False)
test_mask = np.isin(drug_ids, test_drugs)
train_mask = ~test_mask

X_ctl = np.asarray(data["X_ctl"])
y_delta = np.asarray(data["y_delta"])
X_drug = np.asarray(data["X_drug"])

train_ctl = X_ctl[train_mask]
train_trt = y_delta[train_mask]
train_drug = X_drug[train_mask]
train_cells = cell_idx[train_mask]

test_ctl = X_ctl[test_mask]
test_trt = y_delta[test_mask]
test_drug = X_drug[test_mask]
test_cells = cell_idx[test_mask]

print("train sizes:", train_ctl.shape[0], train_drug.shape[0], train_cells.shape[0])
print("test  sizes:", test_ctl.shape[0], test_drug.shape[0], test_cells.shape[0])


train sizes: 14332 14332 14332
test  sizes: 3420 3420 3420


In [6]:
num_genes = int(adj_matrix.shape[0])
model = BaseLineGAT(
    num_genes=num_genes,
    num_cells=num_cells,
    hidden_dim=64,
    num_heads=4,
    dropout=0.2,
    use_residual=False,
    use_drug_embedding=False,
    attention_layer_number=4,
    output_after_embedding=False,
    per_node_embedding=True,
)

loss_mask = tf.constant(data["loss_mask"], dtype=tf.float32)

def pcc_loss(y_true, y_pred):
    mx = tf.reduce_mean(y_true, axis=1, keepdims=True)
    my = tf.reduce_mean(y_pred, axis=1, keepdims=True)
    xm = y_true - mx
    ym = y_pred - my
    r_num = tf.reduce_sum(xm * ym, axis=1)
    r_den = tf.sqrt(tf.reduce_sum(tf.square(xm), axis=1) * tf.reduce_sum(tf.square(ym), axis=1) + 1e-8)
    r = r_num / r_den
    return 1.0 - tf.reduce_mean(r)

def masked_combined_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    mask = tf.cast(loss_mask, tf.float32)
    mse = tf.reduce_sum(tf.square(y_true - y_pred) * mask)
    valid_count = tf.reduce_sum(mask)
    batch_n = tf.cast(tf.shape(y_true)[0], tf.float32)
    mse = mse / tf.maximum(valid_count * batch_n, 1.0)
    valid_indices = tf.where(loss_mask[0] > 0)[:, 0]
    yt = tf.gather(y_true, valid_indices, axis=1)
    yp = tf.gather(y_pred, valid_indices, axis=1)
    pcc = pcc_loss(yt, yp)
    return mse + 5.0 * pcc

class GATWrapper(keras.Model):
    def __init__(self, gat_model, adj_matrix):
        super().__init__()
        self.gat = gat_model
        self.adj = tf.constant(adj_matrix, dtype=tf.float32)

    def call(self, inputs):
        ctl, drug_targets, cell_idx = inputs
        cell_idx = tf.cast(cell_idx, tf.int32)
        return self.gat([self.adj, ctl, drug_targets, cell_idx])

debug_eager = True
if debug_eager:
    tf.config.run_functions_eagerly(True)
    print("debug_eager enabled")

wrapped_model = GATWrapper(model, adj_matrix)
if debug_eager:
    b = min(2, train_ctl.shape[0])
    y0 = wrapped_model([train_ctl[:b], train_drug[:b], train_cells[:b]], training=False)
    print("train_ctl batch:", train_ctl[:b].shape, "y0:", y0.shape, "tf_batch:", int(tf.shape(y0)[0].numpy()))

wrapped_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=masked_combined_loss,
    metrics=[keras.metrics.MeanSquaredError()],
    run_eagerly=debug_eager,
)
print("compiled")


compiled


In [7]:
def eval_pcc_mse(model, ctl, drug, cells, y_true, loss_mask, batch_size=32, max_eval=None):
    if max_eval is not None and len(ctl) > int(max_eval):
        rng = np.random.default_rng(0)
        idx = rng.choice(len(ctl), size=int(max_eval), replace=False)
        ctl = ctl[idx]
        drug = drug[idx]
        cells = cells[idx]
        y_true = y_true[idx]
    pred = model.predict([ctl, drug, cells], batch_size=batch_size, verbose=0)
    valid_indices = np.where(np.asarray(loss_mask)[0] > 0)[0]
    y_true_valid = y_true[:, valid_indices]
    pred_valid = pred[:, valid_indices]
    pcc_list = []
    for i in range(len(y_true_valid)):
        yt = y_true_valid[i]
        yp = pred_valid[i]
        if np.std(yt) > 1e-6 and np.std(yp) > 1e-6:
            p, _ = pearsonr(yt, yp)
            pcc_list.append(p)
    avg_pcc = float(np.mean(pcc_list)) if pcc_list else 0.0
    mse = float(mean_squared_error(y_true_valid, pred_valid))
    return {"mse": mse, "pcc": avg_pcc}

class PCCCallback(keras.callbacks.Callback):
    def __init__(self, loss_mask, train_data, val_data, batch_size=32, max_eval=2048):
        super().__init__()
        self.loss_mask = loss_mask
        self.train_data = train_data
        self.val_data = val_data
        self.batch_size = int(batch_size)
        self.max_eval = int(max_eval) if max_eval is not None else None
        self.train_pcc = []
        self.val_pcc = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        ctl_tr, drug_tr, cell_tr, y_tr = self.train_data
        ctl_va, drug_va, cell_va, y_va = self.val_data
        tr = eval_pcc_mse(self.model, ctl_tr, drug_tr, cell_tr, y_tr, self.loss_mask, batch_size=self.batch_size, max_eval=self.max_eval)
        va = eval_pcc_mse(self.model, ctl_va, drug_va, cell_va, y_va, self.loss_mask, batch_size=self.batch_size, max_eval=self.max_eval)
        self.train_pcc.append(tr["pcc"])
        self.val_pcc.append(va["pcc"])
        logs["pcc"] = tr["pcc"]
        logs["val_pcc"] = va["pcc"]
        print(f"Epoch {epoch+1}: pcc={tr['pcc']:.4f} val_pcc={va['pcc']:.4f}")

pcc_cb = PCCCallback(
    loss_mask=data["loss_mask"],
    train_data=(train_ctl, train_drug, train_cells, train_trt),
    val_data=(test_ctl, test_drug, test_cells, test_trt),
    batch_size=batch_size,
    max_eval=2048,
)


In [ ]:
history = wrapped_model.fit(
    [train_ctl, train_drug, train_cells],
    train_trt,
    epochs=int(epochs),
    batch_size=int(batch_size),
    callbacks=[pcc_cb],
    validation_data=([test_ctl, test_drug, test_cells], test_trt),
)


Epoch 1/20


/opt/anaconda3/envs/RFA_GNN/lib/python3.11/site-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'base_line_gat', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


279/448 ━━━━━━━━━━━━━━━━━━━━ 5:27 2s/step - loss: 6.1580 - mean_squared_error: 1.3033

In [ ]:
train_metrics = eval_pcc_mse(wrapped_model, train_ctl, train_drug, train_cells, train_trt, data["loss_mask"], batch_size=batch_size, max_eval=20000)
test_metrics = eval_pcc_mse(wrapped_model, test_ctl, test_drug, test_cells, test_trt, data["loss_mask"], batch_size=batch_size, max_eval=None)
print(f"Train | MSE: {train_metrics['mse']:.4f} | Sample-wise PCC: {train_metrics['pcc']:.4f}")
print(f"Test  | MSE: {test_metrics['mse']:.4f} | Sample-wise PCC: {test_metrics['pcc']:.4f}")
